# Gold-Standard Sampling Dashboard Realisasi RPJMN

Notebook ini membuat subset **gold-standard** untuk verifikasi manual, dari dataset master indikator RPJMN.

**Pendekatan sampling:** stratified sampling per `sektor`, dengan quota proporsional
(largest remainder method) + oversampling terarah untuk baris yang label `realisasi`
existing-nya `False` (karena jumlahnya sangat minoritas di dataset asli, tapi justru
kasus yang paling penting untuk dievaluasi).

**Output:** subset baris terpilih, dengan kolom `realisasi` dan `implementation_summary`
dikosongkan — siap diisi manual oleh annotator sebagai ground truth.

> Catatan: sampling ini menggunakan label `realisasi` *existing* (yang statusnya draft/unreliable)
> hanya sebagai *sampling frame* untuk menjamin representasi kasus True/False —
> bukan sebagai gold label itu sendiri. Gold label yang sebenarnya tetap harus
> diisi manual berdasarkan bukti (evidence) asli.

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

## Konfigurasi

In [ ]:
INPUT_DIR = Path("../data/raw")
INPUT_FILENAME = "master_rpjmn_202607031500.csv"
INPUT_PATH = INPUT_DIR / INPUT_FILENAME

OUTPUT_DIR = Path("../data/processed")
OUTPUT_FILENAME = "gold_standard_sample.csv"
OUTPUT_PATH_CSV = OUTPUT_DIR / "gold_standard_sample.csv"
OUTPUT_PATH_XLSX = OUTPUT_DIR / "gold_standard_sample.xlsx"

TARGET_N = 292
GROUP_COL = "sektor"
LABEL_COL = "realisasi"
MIN_FALSE_FRACTION = 0.30
RANDOM_SEED = 42

COLUMNS_TO_BLANK = ["realisasi", "implementation_summary"]

## Load data

In [14]:
df = pd.read_csv(INPUT_PATH)
print(f"Total baris: {len(df)}")
df.head()

Total baris: 1459


,id,indikator,sub_indikator,satuan,baseline_2024,target_2025,target_2029,sektor,kl,keywords,realisasi,implementation_summary,created_at,source,source_url,updated_at
0,1,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 04.04.02 - Persentase provinsi dengan indek...,persen,80,"80,85","84,22",Pendidikan,KEMENTERIAN AGAMA,"pemerataan guru, distribusi guru, indeks pemer...",True,"Program berjalan, ditandai program distribusi ...",2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
1,2,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 07.09.02 - Daerah dengan Indeks Inovasi Dae...,daerah (prov/kab/kota),75 (2023),82,147,Birokrasi,KEMENTERIAN DALAM NEGERI,"inovasi daerah, inovatif tinggi, pengembangan ...",True,"Program berjalan, ditandai dengan partisipasi ...",2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
2,3,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 07.09.03 - Jumlah daerah dengan proporsi pa...,daerah (prov/kab/kota),300,300,546,Birokrasi,KEMENTERIAN DALAM NEGERI,"pajak daerah, pendapatan PAD, tax effort, kine...",True,"Program berjalan, ditandai dengan target penin...",2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
3,4,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.09.01 - Indeks Kemajuan Perundingan Bata...,Indeks,52.43,69.75,76.21,Pertahanan,KEMENTERIAN LUAR NEGERI,"batas maritim, perundingan laut, diplomasi mar...",True,"Program berjalan, ditandai dengan kelanjutan p...",2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
4,5,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.22.04 - Produktivitas lahan garam rakyat,ton/ha,83,83,93,Pangan,KEMENTERIAN KELAUTAN DAN PERIKANAN,"produktivitas lahan garam, petani garam, lahan...",True,"Program berjalan, ditandai program peningkatan...",2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000


## Alokasi quota per sektor

Quota dialokasikan proporsional terhadap jumlah baris tiap sektor di dataset asli,
dibulatkan pakai *largest remainder method* supaya totalnya persis `TARGET_N`.

In [15]:
def allocate_quota(df: pd.DataFrame, group_col: str, target_n: int) -> pd.Series:
    counts = df[group_col].value_counts()
    proportions = counts / counts.sum()
    raw = proportions * target_n

    floor_alloc = np.floor(raw).astype(int)
    remainder = target_n - floor_alloc.sum()

    remainders = (raw - floor_alloc).sort_values(ascending=False)
    add_to = remainders.index[:remainder]

    alloc = floor_alloc.copy()
    alloc[add_to] += 1

    # jangan sampai quota > baris yang tersedia di sektor tsb
    alloc = np.minimum(alloc, counts)
    return alloc


quota = allocate_quota(df, GROUP_COL, TARGET_N)
print(quota)
print("\nTotal quota:", quota.sum())

sektor
Sosial           61
Industri         50
Birokrasi        38
Infrastruktur    38
Kesehatan        36
Pendidikan       25
Pangan           21
Pertahanan       15
Energi            8
Name: count, dtype: int64

Total quota: 292


## Sampling per sektor (dengan oversampling kasus False)

Di dalam tiap sektor: ambil dulu porsi `MIN_FALSE_FRACTION` dari quota untuk baris
berlabel `False` (dicap oleh ketersediaan baris False di sektor itu), sisanya diisi
dari baris berlabel `True`. Kalau baris `True` yang tersedia kurang dari kebutuhan,
kekurangannya dialihkan ke `False` (selama masih ada stok).

In [16]:
def sample_group(
    group_df: pd.DataFrame,
    quota_n: int,
    label_col: str,
    min_false_fraction: float,
    rng: np.random.Generator,
) -> pd.DataFrame:
    false_pool = group_df[group_df[label_col] == False]
    true_pool = group_df[group_df[label_col] == True]

    target_false = min(len(false_pool), round(quota_n * min_false_fraction))
    target_true = quota_n - target_false

    if target_true > len(true_pool):
        deficit = target_true - len(true_pool)
        target_true = len(true_pool)
        target_false = min(len(false_pool), target_false + deficit)

    sampled_false = (
        false_pool.sample(n=target_false, random_state=rng.integers(0, 1_000_000))
        if target_false > 0 else false_pool.iloc[0:0]
    )
    sampled_true = (
        true_pool.sample(n=target_true, random_state=rng.integers(0, 1_000_000))
        if target_true > 0 else true_pool.iloc[0:0]
    )
    return pd.concat([sampled_false, sampled_true])


rng = np.random.default_rng(RANDOM_SEED)

samples = []
for sektor, quota_n in quota.items():
    group_df = df[df[GROUP_COL] == sektor]
    sampled = sample_group(group_df, int(quota_n), LABEL_COL, MIN_FALSE_FRACTION, rng)
    samples.append(sampled)

gold_sample = pd.concat(samples).sort_values("id").reset_index(drop=True)
print("Ukuran sample akhir:", len(gold_sample))

Ukuran sample akhir: 292


## Diagnostik distribusi sample

In [17]:
print("Per sektor:")
print(gold_sample[GROUP_COL].value_counts(), "\n")

print("Per label (existing, sblm dikosongkan):")
print(gold_sample[LABEL_COL].value_counts(), "\n")

print("Per sektor x label:")
print(gold_sample.groupby([GROUP_COL, LABEL_COL]).size())

Per sektor:
sektor
Sosial           61
Industri         50
Infrastruktur    38
Birokrasi        38
Kesehatan        36
Pendidikan       25
Pangan           21
Pertahanan       15
Energi            8
Name: count, dtype: int64 

Per label (existing, sblm dikosongkan):
realisasi
True     220
False     72
Name: count, dtype: int64 

Per sektor x label:
sektor         realisasi
Birokrasi      False        11
               True         27
Energi         False         1
               True          7
Industri       False        15
               True         35
Infrastruktur  False         6
               True         32
Kesehatan      False         3
               True         33
Pangan         False         6
               True         15
Pendidikan     False         8
               True         17
Pertahanan     False         4
               True         11
Sosial         False        18
               True         43
dtype: int64


## Kosongkan kolom untuk diisi ulang manual (gold label)

`realisasi` dan `implementation_summary` dikosongkan di sini. Kolom lain
(termasuk `source` / `source_url` existing) tetap dipertahankan apa adanya —
tapi ingat, `source_url` existing di dataset ini semuanya menunjuk ke dokumen
Perpres RPJMN itu sendiri, **bukan bukti realisasi**. Saat annotator mengisi ulang
`realisasi`/`implementation_summary` secara manual, sebaiknya `source`/`source_url`
juga diupdate ke evidence yang benar-benar dipakai (bukan disalin dari nilai lama).

Kalau kamu mau kolom itu ikut dikosongkan otomatis juga, tinggal tambahkan
`"source"` dan `"source_url"` ke `COLUMNS_TO_BLANK` di cell konfigurasi di atas.

In [18]:
gold_sample[COLUMNS_TO_BLANK] = pd.NA
gold_sample.head()

,id,indikator,sub_indikator,satuan,baseline_2024,target_2025,target_2029,sektor,kl,keywords,realisasi,implementation_summary,created_at,source,source_url,updated_at
0,12,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.12.08 - Kabupaten/kota yang mendeklarasi...,kab/kota,4 (2023),30,200,Kesehatan,KEMENTERIAN KESEHATAN,"STBM 5 pilar, deklarasi STBM, sanitasi total, ...",<NA>,<NA>,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
1,13,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.09.03 - Persentase kebijakan di bidang k...,persen,0,20,100,Pertahanan,KEMENTERIAN LUAR NEGERI,"kerja sama, prioritas nasional, kebijakan glob...",<NA>,<NA>,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
2,15,Pengampu Pencapaian Indikator Kegiatan Prioritas,"KP 02.09.04 - Persentase keberhasilan promosi,...",persen,75,80,82,Industri,KEMENTERIAN LUAR NEGERI,"promosi ekonomi, norma internasional, kesepaka...",<NA>,<NA>,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
3,19,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 07.16.02 - Indeks Diplomasi Perlindungan WN...,Indeks,98.5,98.7,99.1,Sosial,KEMENTERIAN LUAR NEGERI,"diplomasi WNI, perlindungan WNI, hubungan luar...",<NA>,<NA>,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
4,20,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.01.01 - Persentase pemenuhan alutsista,persen,30.3,30.3,100,Pertahanan,KEMENTERIAN PERTAHANAN,"pemenuhan alutsista, modernisasi senjata, pera...",<NA>,<NA>,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000


## Simpan output

In [25]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
gold_sample.to_csv(OUTPUT_PATH_CSV, index=False)
gold_sample.to_excel(OUTPUT_PATH_XLSX, index=False, engine="openpyxl")

print(f"Tersimpan CSV : {OUTPUT_PATH_CSV.resolve()}")
print(f"Tersimpan XLSX: {OUTPUT_PATH_XLSX.resolve()}")
print(f"Jumlah baris  : {len(gold_sample)}")
print(f"Kolom         : {list(gold_sample.columns)}")

Tersimpan CSV : C:\Users\Aufii\Documents\Informatika\OTHERS\10. PKL 2026\1. code\4. project rpjmn\data\processed\gold_standard_sample.csv
Tersimpan XLSX: C:\Users\Aufii\Documents\Informatika\OTHERS\10. PKL 2026\1. code\4. project rpjmn\data\processed\gold_standard_sample.xlsx
Jumlah baris  : 292
Kolom         : ['id', 'indikator', 'sub_indikator', 'satuan', 'baseline_2024', 'target_2025', 'target_2029', 'sektor', 'kl', 'keywords', 'realisasi', 'implementation_summary', 'created_at', 'source', 'source_url', 'updated_at']


In [27]:
output = pd.read_csv(OUTPUT_PATH_CSV)
output.head()

,id,indikator,sub_indikator,satuan,baseline_2024,target_2025,target_2029,sektor,kl,keywords,realisasi,implementation_summary,created_at,source,source_url,updated_at
0,12,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.12.08 - Kabupaten/kota yang mendeklarasi...,kab/kota,4 (2023),30,200,Kesehatan,KEMENTERIAN KESEHATAN,"STBM 5 pilar, deklarasi STBM, sanitasi total, ...",NaN,NaN,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
1,13,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.09.03 - Persentase kebijakan di bidang k...,persen,0,20,100,Pertahanan,KEMENTERIAN LUAR NEGERI,"kerja sama, prioritas nasional, kebijakan glob...",NaN,NaN,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
2,15,Pengampu Pencapaian Indikator Kegiatan Prioritas,"KP 02.09.04 - Persentase keberhasilan promosi,...",persen,75,80,82,Industri,KEMENTERIAN LUAR NEGERI,"promosi ekonomi, norma internasional, kesepaka...",NaN,NaN,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
3,19,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 07.16.02 - Indeks Diplomasi Perlindungan WN...,Indeks,98.5,98.7,99.1,Sosial,KEMENTERIAN LUAR NEGERI,"diplomasi WNI, perlindungan WNI, hubungan luar...",NaN,NaN,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000
4,20,Pengampu Pencapaian Indikator Kegiatan Prioritas,KP 02.01.01 - Persentase pemenuhan alutsista,persen,30.3,30.3,100,Pertahanan,KEMENTERIAN PERTAHANAN,"pemenuhan alutsista, modernisasi senjata, pera...",NaN,NaN,2025-12-10 04:20:46.000,jdih.kemenkoinfra.go.id,https://jdih.kemenkoinfra.go.id/cfind/source/f...,2025-02-10 00:00:00.000


In [28]:
output.info()

<class 'pandas.DataFrame'>
RangeIndex: 292 entries, 0 to 291
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      292 non-null    int64  
 1   indikator               292 non-null    str    
 2   sub_indikator           292 non-null    str    
 3   satuan                  292 non-null    str    
 4   baseline_2024           292 non-null    str    
 5   target_2025             292 non-null    str    
 6   target_2029             291 non-null    str    
 7   sektor                  292 non-null    str    
 8   kl                      292 non-null    str    
 9   keywords                292 non-null    str    
 10  realisasi               0 non-null      float64
 11  implementation_summary  0 non-null      float64
 12  created_at              292 non-null    str    
 13  source                  292 non-null    str    
 14  source_url              292 non-null    str    
 15  